In [8]:
import pandas as pd
import re
from pathlib import Path

main_file = Path(r"D:\Proyek FEB\user 2\data_2026_sitasi full.xlsx")
nan_file = Path(r"data_nan.xlsx")

df_main = pd.read_excel(main_file)
df_nan = pd.read_excel(nan_file)

COL_NO = "No"
COL_TITLE = "Title"
COL_JOURNAL = "Journal"
COL_Q = "SCOPUS Q"

In [30]:
def is_empty_value(value):
    if pd.isna(value):
        return True
    
    text = str(value).strip()
    
    return text == "" or text.lower() in [
        "nan", "none", "null", "-", "?",
        "??", "???", "n/a", "na",
        "tidak ada", "not found"
    ]


def normalize_text(value):
    if pd.isna(value):
        return ""
    
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    
    return text


def title_key(value):
    if pd.isna(value):
        return ""
    
    text = str(value).lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    
    return text.strip()


def is_book_or_buku(value):
    if pd.isna(value):
        return False
    
    text = str(value).lower().strip()
    
    return bool(re.search(r"\b(book|buku)\b", text))

In [31]:
mask_book_nan = df_nan[COL_JOURNAL].apply(is_book_or_buku)

print("Jumlah baris book/buku di data_nan:", mask_book_nan.sum())

df_nan.loc[mask_book_nan, COL_Q] = "no-Q"

Jumlah baris book/buku di data_nan: 44


In [32]:
df_main["merge_key"] = (
    df_main[COL_TITLE].apply(normalize_text) + " || " +
    df_main[COL_JOURNAL].apply(normalize_text)
)

df_nan["merge_key"] = (
    df_nan[COL_TITLE].apply(normalize_text) + " || " +
    df_nan[COL_JOURNAL].apply(normalize_text)
)

In [33]:
df_nan_filled = df_nan[~df_nan[COL_Q].apply(is_empty_value)].copy()

print("Jumlah baris data_nan yang Q-nya terisi:", len(df_nan_filled))

Jumlah baris data_nan yang Q-nya terisi: 183


In [34]:
conflict_check = (
    df_nan_filled
    .groupby("merge_key")[COL_Q]
    .nunique()
    .reset_index(name="unique_q_count")
)

conflict_keys = set(
    conflict_check[conflict_check["unique_q_count"] > 1]["merge_key"]
)

print("Jumlah konflik Title + Journal:", len(conflict_keys))

Jumlah konflik Title + Journal: 1


In [35]:
conflict_detail = (
    df_nan_filled[df_nan_filled["merge_key"].isin(conflict_keys)]
    [[COL_TITLE, COL_JOURNAL, COL_Q, "merge_key"]]
    .drop_duplicates()
    .sort_values([COL_TITLE, COL_JOURNAL])
)

conflict_detail

,Title,Journal,SCOPUS Q,merge_key
124,Islamic and conventional bank deposits: what d...,International Journal of Islamic and Middle Ea...,Q2,islamic and conventional bank deposits: what d...
163,Islamic and conventional bank deposits: what d...,International Journal of Islamic and Middle Ea...,Q1,islamic and conventional bank deposits: what d...


In [36]:
if len(conflict_detail) > 0:
    conflict_key = conflict_detail.iloc[0]["merge_key"]
    
    df_nan_filled.loc[
        df_nan_filled["merge_key"] == conflict_key,
        COL_Q
    ] = "Q2"

    print("Konflik pertama sudah dioverride menjadi Q2")

Konflik pertama sudah dioverride menjadi Q2


In [37]:
conflict_check = (
    df_nan_filled
    .groupby("merge_key")[COL_Q]
    .nunique()
    .reset_index(name="unique_q_count")
)

conflict_keys = set(
    conflict_check[conflict_check["unique_q_count"] > 1]["merge_key"]
)

print("Jumlah konflik setelah override:", len(conflict_keys))

Jumlah konflik setelah override: 0


In [38]:
df_nan_safe = df_nan_filled[
    ~df_nan_filled["merge_key"].isin(conflict_keys)
].copy()

q_map = (
    df_nan_safe
    .drop_duplicates(subset=["merge_key"])
    .set_index("merge_key")[COL_Q]
    .to_dict()
)

print("Jumlah mapping aman Title + Journal:", len(q_map))

Jumlah mapping aman Title + Journal: 125


In [39]:
updated_count = 0
same_count = 0
not_found_count = 0

for idx, row in df_main.iterrows():
    key = row["merge_key"]
    
    if key in q_map:
        old_q = df_main.at[idx, COL_Q]
        new_q = q_map[key]
        
        df_main.at[idx, COL_Q] = new_q
        
        if str(old_q).strip() != str(new_q).strip():
            updated_count += 1
        else:
            same_count += 1
    else:
        not_found_count += 1

print("Jumlah SCOPUS Q yang diupdate dari data_nan:", updated_count)
print("Jumlah SCOPUS Q yang sudah sama:", same_count)
print("Jumlah baris data utama yang tidak ada di data_nan:", not_found_count)

Jumlah SCOPUS Q yang diupdate dari data_nan: 0
Jumlah SCOPUS Q yang sudah sama: 180
Jumlah baris data utama yang tidak ada di data_nan: 2099


In [40]:
manual_remaining_q_raw = {
    "Islamic fintech and ESG goals: Key considerations for fulfilling maqasid principles": "no-Q",
    
    "Literature Review of Digital Transformation on Banking: Corporate Perspective": "no-Q",
    
    "Interactions among the primary causes of carbon dioxide emissions in selected south Asian countries: Does renewable energy mitigate carbon dioxide emissions?": "Q3",
    
    "Transformation to Enhance Business Resilience In Construction Companies: Systematic Literature Review Of Dimensional Models": "no-Q",
    
    "Human resource management in the digital era: Developing digital-first organizational cultures to drive sustainable competitive advantage": "no-Q",
    
    "Trends and implications of islamic bank financing in Indonesia: A comparative analysis of profit-loss sharing and non-profit-loss sharing contracts (2006-2023)": "no-Q",
    
    "Trust in Government and Tax Compliance in Indonesia and Malaysia: Do Ethics and Tax Amnesty Matter?": "no-Q",
    
    "Integrating Waqf-Based Forests and Carbon Trading: Opportunities, Challenges, and Strategies in Indonesia": "Q2",
    
    "Corporate Resilience to Recover from Shocks: The Role of Corporate Social Responsibility and Corporate Reputation": "Q4",
}

In [41]:
manual_remaining_q = {
    title_key(title): q
    for title, q in manual_remaining_q_raw.items()
}

df_main["title_key_manual"] = df_main[COL_TITLE].apply(title_key)

manual_remaining_count = 0

for idx, row in df_main.iterrows():
    key = row["title_key_manual"]
    
    if key in manual_remaining_q:
        old_q = df_main.at[idx, COL_Q]
        new_q = manual_remaining_q[key]
        
        df_main.at[idx, COL_Q] = new_q
        
        if str(old_q).strip() != str(new_q).strip():
            manual_remaining_count += 1

print("Jumlah baris yang diisi dari manual remaining:", manual_remaining_count)

Jumlah baris yang diisi dari manual remaining: 0


In [42]:
df_main[COL_Q].value_counts(dropna=False)

SCOPUS Q
Q1      714
Q2      607
no-Q    530
Q3      293
Q4      135
Name: count, dtype: int64

In [43]:
remaining_empty = df_main[df_main[COL_Q].apply(is_empty_value)].copy()

print("Sisa SCOPUS Q kosong:", len(remaining_empty))

remaining_unique = (
    remaining_empty[[COL_TITLE, COL_JOURNAL, COL_Q]]
    .drop_duplicates()
    .sort_values([COL_JOURNAL, COL_TITLE])
    .reset_index(drop=True)
)

remaining_unique

Sisa SCOPUS Q kosong: 0


,Title,Journal,SCOPUS Q


In [45]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2279 entries, 0 to 2278
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   No                    2279 non-null   int64  
 1   Title                 2279 non-null   object 
 2   Authors               2279 non-null   object 
 3   Journal               2271 non-null   object 
 4   SCOPUS Q              2279 non-null   object 
 5   Tahun                 2278 non-null   object 
 6   Volume / Issue        2171 non-null   object 
 7   LINK DOI/ARTIKEL      2279 non-null   object 
 8   SCOPUS SITASI         2279 non-null   int64  
 9   raw_key               2145 non-null   object 
 10  q_source_checkpoint2  2153 non-null   object 
 11  journal_key_strong    2143 non-null   object 
 12  q_empty               2153 non-null   float64
 13  merge_key             2279 non-null   object 
 14  title_key_manual      2279 non-null   object 
dtypes: float64(1), int64(

In [46]:
df_main.drop(columns=["raw_key", "q_source_checkpoint2","journal_key_strong","q_empty","merge_key","title_key_manual"], inplace=True)
df_main.head()

,No,Title,Authors,Journal,SCOPUS Q,Tahun,Volume / Issue,LINK DOI/ARTIKEL,SCOPUS SITASI
0,1,Ethnic identity and internal migration decisio...,ILMIAWAN AUWALIN,Journal of Ethnic and Migration Studies,Q1,2019-01-11 00:00:00,"Vol. 14, Issue 13",10.1080/1369183X.2018.1561252,24
1,2,The role of service quality within Indonesian ...,BADRI MUNIR SUKOCO,Journal of Islamic Marketing,Q2,2020-01-14 00:00:00,"Vol. 11, Issue 1",10.1108/JIMA-03-2017-0033,60
2,3,Developing Islamic crowdfunding website platfo...,MUHAMAD NAFIK HADI RYANDONO,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48
3,4,Developing Islamic crowdfunding website platfo...,ACHSANIA HENDRATMI,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48
4,5,Developing Islamic crowdfunding website platfo...,PUJI SUCIA SUKMANINGRUM,Journal of Islamic Marketing,Q2,2020-08-21 00:00:00,"Vol. 11, Issue 5",10.1108/JIMA-02-2019-0022,48


In [47]:
df_main.to_excel("data_full_sqopus_sitasi.xlsx", index=False)